# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates loading and processing the FAIR² dataset provided with a Croissant schema, using the `mlcroissant` library. All references to record sets, fields, or columns use their `@id` from the dataset.

### Dataset Source
The dataset is described by its Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant for this environment
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset loaded: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview

List the record sets available in the dataset, along with their `@id`, their fields, and field `@id`s. This step helps identify all accessible information for further analysis.

In [ ]:
# List record sets and their fields using their @id values.

record_sets = list(dataset.record_sets)

print('Available Record Sets:')
for rs in record_sets:
    print(f"- Record Set name: {rs['name']}, @id: {rs['@id']}")
    for field in rs.get('field', []):
        # Each field is a dict
        print(f"    - Field: {field['name']}, @id: {field['@id']}")

## 3. Data Extraction

Load data from each available record set into a DataFrame using the record set and field `@id`s. This approach will allow support for multiple tables within this Croissant package.

In [ ]:
# Extract all data to pandas DataFrames using record set and field @ids
dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if not records:
        print(f"No records found for record set @id: {rs_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"---\nLoaded {len(df)} records from record set: {rs_id}")
    print(f"Columns (field @ids): {list(df.columns)}\n")
# Example: display the first record set DataFrame, if available
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"Sample data from record set {first_rs_id}:")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Explore the content of the main data table. We will select a numeric field (using its `@id`), filter, and normalize it, and demonstrate grouping by a relevant field.

In [ ]:
# Choose a record set ID and fields based on the record sets discovered above
# Replace these @ids after running the data overview step above to match your data

# Example values -- update these values with your dataset's entities
record_set_id = None
numeric_field_id = None
group_field_id = None

if dataframes:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    print(f"Fields in this record set: {list(df.columns)}")
    # Try to select a numeric column automatically for demonstration
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is not None:
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with @id '{numeric_field_id}' > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized field '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Pick a categorical field for grouping (non-numeric and not the index)
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field_id = col
                break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print('No suitable group field found for grouping operation.')
    else:
        print('No numeric field detected in the example record set.')
else:
    print("No data available to analyze. Ensure your record set IDs are correct.")

## 5. Visualization

We'll plot the distribution of the selected numeric field, and if grouping was done above, a bar plot of group means. These use the record set and field `@id`s discovered above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of field {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion

In this notebook, we loaded, inspected, and processed data from a Croissant-compatible dataset using the `mlcroissant` library. All dataset elements were referenced by their `@id`, ensuring reproducible and standards-based access. This workflow supports scalable and transparent data exploration and can be adapted for further statistical or machine learning analysis.

Replace example `@id` values above with those from your specific FAIR² dataset as needed.